In [2]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.layers import Input, Dense, ReLU, Conv1D, GlobalAveragePooling1D
from keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
import os
import time

# --- Configuration ---
print("TensorFlow Version:", tf.__version__)

# Model filenames
MLP_FILE = 'model_mlp.h5'
TCN_FILE = 'model_tcn.h5'
ESN_FILE = 'model_esn.npz'

# Training parameters
EPOCHS = 200
BATCH_SIZE = 64
VALIDATION_SPLIT = 0.1
TCN_LOOKBACK = 4  # How many past steps the TCN looks at

# --- 1. Load Dataset ---
print(f"Loading dataset from pucs_dataset.npz...")
try:
    with np.load('pucs_dataset.npz') as data:
        # --- THIS IS THE FIX ---
        # Convert data to float32 to match the Keras model's default type
        X_train_full = data['X_train'].astype(np.float32)
        y_train_full = data['y_train'].astype(np.float32)
        # ---------------------
        
        data_min = data['data_min']
        data_max = data['data_max']
except FileNotFoundError:
    print("ERROR: pucs_dataset.npz not found.")
    print("Please run the generate_pucs.py script first.")
    raise

print(f"Full dataset shape: X={X_train_full.shape}, y={y_train_full.shape}")
print("--- Setup Cell Complete ---")

TensorFlow Version: 2.20.0
Loading dataset from pucs_dataset.npz...
Full dataset shape: X=(100000, 3), y=(100000, 3)
--- Setup Cell Complete ---


In [3]:
print("\n--- Training Model 1: Tiny-MLP (with Tanh) ---")

# --- Custom Training Step with Scheduled Sampling ---
@tf.function
def train_step_scheduled_mlp(x_batch, y_batch, model, optimizer, loss_fn, schedule_prob=0.1):
    with tf.GradientTape() as tape:
        y_pred = model(x_batch)
        mask = tf.random.uniform(shape=y_pred.shape) > schedule_prob
        recursive_input = tf.where(mask, y_batch, y_pred)
        y_pred_recursive = model(recursive_input)
        
        loss_open_loop = loss_fn(y_batch, y_pred)
        loss_recursive = loss_fn(y_batch, y_pred_recursive)
        total_loss = (loss_open_loop + loss_recursive) / 2.0

    gradients = tape.gradient(total_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return total_loss

# --- Model Definition (FIXED) ---
def build_mlp():
    x_in = Input(shape=(3,))
    # --- FIX: Use 'tanh' to get a balanced (non-biased) tap ---
    x = Dense(4, activation='tanh', name="hidden_layer")(x_in) 
    x_out = Dense(3, name="output_layer")(x)
    return Model(x_in, x_out, name="Tiny_MLP")

# --- Data Preparation ---
X_train_mlp, X_val_mlp, y_train_mlp, y_val_mlp = train_test_split(
    X_train_full, y_train_full, test_size=VALIDATION_SPLIT, random_state=42
)

train_ds_mlp = tf.data.Dataset.from_tensor_slices((X_train_mlp.astype(np.float32), y_train_mlp.astype(np.float32))).batch(BATCH_SIZE).shuffle(1000)
val_ds_mlp = tf.data.Dataset.from_tensor_slices((X_val_mlp.astype(np.float32), y_val_mlp.astype(np.float32))).batch(BATCH_SIZE)

# --- Build and Compile ---
model_mlp = build_mlp()
model_mlp.summary()
optimizer_mlp = keras.optimizers.Adam(learning_rate=1e-3)
loss_fn_mlp = keras.losses.MeanSquaredError()
model_mlp.compile(optimizer=optimizer_mlp, loss=loss_fn_mlp) # For .evaluate()

# --- Custom Training Loop ---
print(f"Starting MLP training for {EPOCHS} epochs...")
for epoch in range(EPOCHS):
    start_time = time.time()
    total_loss = 0
    
    for step, (x_batch, y_batch) in enumerate(train_ds_mlp):
        loss = train_step_scheduled_mlp(x_batch, y_batch, model_mlp, optimizer_mlp, loss_fn_mlp, schedule_prob=0.1)
        total_loss += loss

    # Show validation loss every 10 epochs
    if (epoch + 1) % 10 == 0:
        val_loss = model_mlp.evaluate(val_ds_mlp, verbose=0)
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/len(train_ds_mlp):.6f} - Val_Loss: {val_loss:.6f} - Time: {time.time()-start_time:.2f}s")

# --- Save Model ---
model_mlp.save(MLP_FILE)
print(f"Tiny-MLP model saved to {MLP_FILE}")
print("--- MLP Cell Complete ---")


--- Training Model 1: Tiny-MLP ---


Model: "Tiny_MLP"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 3)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ hidden_layer (Dense)                 │ (None, 4)                   │              16 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ re_lu (ReLU)                         │ (None, 4)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ output_layer (Dense)                 │ (None, 3)                   │              15 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 31 (124.00 B)

 Trainable params: 31 (124.00 B)

 Non-trainable params: 0 (0.00 B)

Starting MLP training for 200 epochs...
Epoch 10/200 - Loss: 0.000003 - Val_Loss: 0.000004 - Time: 1.49s
Epoch 20/200 - Loss: 0.000003 - Val_Loss: 0.000004 - Time: 1.46s
Epoch 30/200 - Loss: 0.000003 - Val_Loss: 0.000004 - Time: 1.45s
Epoch 40/200 - Loss: 0.000003 - Val_Loss: 0.000004 - Time: 1.55s
Epoch 50/200 - Loss: 0.000003 - Val_Loss: 0.000004 - Time: 1.48s
Epoch 60/200 - Loss: 0.000003 - Val_Loss: 0.000005 - Time: 1.46s
Epoch 70/200 - Loss: 0.000003 - Val_Loss: 0.000004 - Time: 1.50s
Epoch 80/200 - Loss: 0.000003 - Val_Loss: 0.000004 - Time: 1.45s
Epoch 90/200 - Loss: 0.000003 - Val_Loss: 0.000004 - Time: 1.46s
Epoch 100/200 - Loss: 0.000003 - Val_Loss: 0.000004 - Time: 1.52s
Epoch 110/200 - Loss: 0.000003 - Val_Loss: 0.000004 - Time: 1.54s
Epoch 120/200 - Loss: 0.000003 - Val_Loss: 0.000004 - Time: 1.53s
Epoch 130/200 - Loss: 0.000003 - Val_Loss: 0.000005 - Time: 1.45s
Epoch 140/200 - Loss: 0.000003 - Val_Loss: 0.000004 - Time: 1.40s
Epoch 150/200 - Loss: 0.000003 - Val_Loss: 0.

Epoch 200/200 - Loss: 0.000003 - Val_Loss: 0.000004 - Time: 1.58s
Tiny-MLP model saved to model_mlp.h5
--- MLP Cell Complete ---


In [4]:
print("\n--- Training Model 2: Micro-ESN ---")
# ESNs are trained differently. We create a fixed reservoir,
# push all data through it, and then just train a simple linear output layer.

# --- ESN Parameters ---
N_RESERVOIR = 20  # Number of recurrent "reservoir" neurons
SPECTRAL_RADIUS = 0.95  # Controls dynamics (<= 1 for stability)
SPARSITY = 0.1  # Percentage of connections in reservoir

# --- Create Reservoir Matrices (Fixed and Random) ---
np.random.seed(42) # for reproducibility
W_in = np.random.rand(N_RESERVOIR, 3) * 2 - 1  # Input weights

W_res = np.random.rand(N_RESERVOIR, N_RESERVOIR) * 2 - 1
W_res[np.random.rand(*W_res.shape) > SPARSITY] = 0  # Make sparse
radius = np.max(np.abs(np.linalg.eigvals(W_res)))
W_res = W_res * (SPECTRAL_RADIUS / radius)  # Rescale spectral radius

# --- Run data through the reservoir to get states ---
print("Calculating ESN reservoir states...")
n_samples = X_train_full.shape[0]
reservoir_states = np.zeros((n_samples, N_RESERVOIR))
current_state = np.zeros(N_RESERVOIR)

for i in range(n_samples):
    x_in = X_train_full[i]
    # ESN update equation (hardware-friendly)
    # Using tanh here as it's the standard for ESN state calculation.
    # This state logic will be implemented in hardware.
    current_state = np.tanh(W_in @ x_in + W_res @ current_state)
    reservoir_states[i] = current_state

# --- Train the output layer ---
print("Training ESN output layer (Ridge Regression)...")
ridge_solver = Ridge(alpha=1e-6, fit_intercept=True)
ridge_solver.fit(reservoir_states, y_train_full)

# Get trained weights and biases
W_out = ridge_solver.coef_  # Shape (3, N_RESERVOIR)
b_out = ridge_solver.intercept_ # Shape (3,)

# --- Save Model ---
np.savez(ESN_FILE, W_in=W_in, W_res=W_res, W_out=W_out, b_out=b_out, data_min=data_min, data_max=data_max)
print(f"Micro-ESN model matrices saved to {ESN_FILE}")
print("--- ESN Cell Complete ---")


--- Training Model 2: Micro-ESN ---
Calculating ESN reservoir states...
Training ESN output layer (Ridge Regression)...
Micro-ESN model matrices saved to model_esn.npz
--- ESN Cell Complete ---


In [5]:
from sklearn.metrics import mean_squared_error

print("Calculating ESN Mean Squared Error...")

# Get the ESN's predictions on the entire training dataset
y_pred_esn = ridge_solver.predict(reservoir_states)

# Calculate the MSE by comparing predictions to the true values
mse_esn = mean_squared_error(y_train_full, y_pred_esn)

print(f"--- ESN Training Results ---")
print(f"Mean Squared Error (MSE) on training data: {mse_esn:.10f}")

Calculating ESN Mean Squared Error...
--- ESN Training Results ---
Mean Squared Error (MSE) on training data: 0.0000127358


In [8]:
EPOCHS = 30
print("\n--- Training Model 3: Tiny-TCN (with Tanh) ---")

# --- Model Definition (FIXED) ---
def build_tcn(lookback=TCN_LOOKBACK):
    x_in = Input(shape=(lookback, 3))
    # --- FIX: Use 'tanh' to get a balanced (non-biased) tap ---
    x = Conv1D(filters=8, kernel_size=2, padding='causal', activation='tanh', name="conv1")(x_in)
    x = Conv1D(filters=8, kernel_size=2, padding='causal', activation='tanh', name="conv2")(x)
    
    x = tf.keras.layers.Lambda(lambda t: t[:, -1, :], name='get_last_timestep')(x)
    x_out = Dense(3, name="output_layer")(x)
    return Model(x_in, x_out, name="Tiny_TCN")

# --- Data Preparation ---
def create_sequences(X, y, lookback=TCN_LOOKBACK):
    X_seq, y_seq = [], []
    for i in range(len(X) - lookback):
        X_seq.append(X[i:(i + lookback)])
        y_seq.append(y[i + lookback - 1]) 
    return np.array(X_seq), np.array(y_seq)

print(f"Creating sequences with lookback={TCN_LOOKBACK}...")
X_seq, y_seq = create_sequences(X_train_full, y_train_full, TCN_LOOKBACK)

X_train_tcn, X_val_tcn, y_train_tcn, y_val_tcn = train_test_split(
    X_seq, y_seq, test_size=VALIDATION_SPLIT, random_state=42
)

# Ensure data is float32
train_ds_tcn = tf.data.Dataset.from_tensor_slices((X_train_tcn.astype(np.float32), y_train_tcn.astype(np.float32))).batch(BATCH_SIZE).shuffle(1000)
val_ds_tcn = tf.data.Dataset.from_tensor_slices((X_val_tcn.astype(np.float32), y_val_tcn.astype(np.float32))).batch(BATCH_SIZE)

# --- Build and Compile ---
model_tcn = build_tcn()
model_tcn.summary()
optimizer_tcn = keras.optimizers.Adam(learning_rate=1e-3)
loss_fn_tcn = keras.losses.MeanSquaredError()
model_tcn.compile(optimizer=optimizer_tcn, loss=loss_fn_tcn) # For .evaluate()

# --- Standard Training Loop ---
print(f"Starting TCN training for {EPOCHS} epochs...")
for epoch in range(EPOCHS):
    start_time = time.time()
    total_loss = 0
    
    for step, (x_batch, y_batch) in enumerate(train_ds_tcn):
        with tf.GradientTape() as tape:
            y_pred = model_tcn(x_batch, training=True)
            loss = loss_fn_tcn(y_batch, y_pred)
        
        gradients = tape.gradient(loss, model_tcn.trainable_variables)
        optimizer_tcn.apply_gradients(zip(gradients, model_tcn.trainable_variables))
        total_loss += loss

    # Show validation loss every 10 epochs
    if (epoch + 1) % 10 == 0:
        val_loss = model_tcn.evaluate(val_ds_tcn, verbose=0)
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/len(train_ds_tcn):.6f} - Val_Loss: {val_loss:.6f} - Time: {time.time()-start_time:.2f}s")

# --- Save Model ---
model_tcn.save(TCN_FILE)
print(f"Tiny-TCN model saved to {TCN_FILE}")
print("--- TCN Cell Complete ---")


--- Training Model 3: Tiny-TCN ---
Creating sequences with lookback=4...


Model: "Tiny_TCN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)           │ (None, 4, 3)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1 (Conv1D)                       │ (None, 4, 8)                │              56 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2 (Conv1D)                       │ (None, 4, 8)                │             136 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ get_last_timestep (Lambda)           │ (None, 8)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ output_layer (Dense)                 │ (None, 3)                   │              27 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 219 (876.00 B)

 Trainable params: 219 (876.00 B)

 Non-trainable params: 0 (0.00 B)

Starting TCN training for 30 epochs...
Epoch 1/30 - Loss: 0.007642 - Val_Loss: 0.000127 - Time: 38.37s
Epoch 2/30 - Loss: 0.000043 - Val_Loss: 0.000006 - Time: 38.89s
Epoch 3/30 - Loss: 0.000005 - Val_Loss: 0.000003 - Time: 37.32s
Epoch 4/30 - Loss: 0.000004 - Val_Loss: 0.000003 - Time: 38.33s
Epoch 5/30 - Loss: 0.000003 - Val_Loss: 0.000003 - Time: 38.67s
Epoch 6/30 - Loss: 0.000003 - Val_Loss: 0.000002 - Time: 39.35s
Epoch 7/30 - Loss: 0.000003 - Val_Loss: 0.000002 - Time: 39.24s
Epoch 8/30 - Loss: 0.000002 - Val_Loss: 0.000002 - Time: 38.74s
Epoch 9/30 - Loss: 0.000002 - Val_Loss: 0.000002 - Time: 37.47s
Epoch 10/30 - Loss: 0.000002 - Val_Loss: 0.000003 - Time: 38.39s
Epoch 11/30 - Loss: 0.000002 - Val_Loss: 0.000002 - Time: 38.38s
Epoch 12/30 - Loss: 0.000002 - Val_Loss: 0.000003 - Time: 36.82s
Epoch 13/30 - Loss: 0.000002 - Val_Loss: 0.000001 - Time: 38.65s
Epoch 14/30 - Loss: 0.000002 - Val_Loss: 0.000001 - Time: 38.50s
Epoch 15/30 - Loss: 0.000002 - Val_Loss: 0.000001 - Time: 38

Epoch 30/30 - Loss: 0.000001 - Val_Loss: 0.000001 - Time: 38.71s
Tiny-TCN model saved to model_tcn.h5
--- TCN Cell Complete ---


In [9]:
from sklearn.linear_model import LinearRegression

print("\n--- Training Model 4: AutoRegressive (AR) Model ---")

# --- Data Preparation for AR(2) Model ---
# We want to predict y_train_full[i] based on:
# X_train_full[i] (current state) AND X_train_full[i-1] (previous state)

AR_LOOKBACK = 2 # AR(2) model

# Create the input features [X_n, X_{n-1}]
# We start from i=1 so we have a past value
X_ar_features = np.hstack([
    X_train_full[AR_LOOKBACK-1:],  # Current state (X_n)
    X_train_full[:-AR_LOOKBACK+1]  # Previous state (X_{n-1})
])

# The target is the state at n+1 (which is y_train_full[n])
y_ar_target = y_train_full[AR_LOOKBACK-1:]

print(f"AR Model Input Shape: {X_ar_features.shape}") # (99999, 6)
print(f"AR Model Target Shape: {y_ar_target.shape}") # (99999, 3)

# --- Train the AR Model ---
print("Training AR(2) linear model...")
# A simple linear regression is all we need
ar_model = LinearRegression()
ar_model.fit(X_ar_features, y_ar_target)

# Get the trained weights and biases
W_ar = ar_model.coef_       # Shape (3, 6)
b_ar = ar_model.intercept_  # Shape (3,)

# --- Save Model ---
AR_FILE = 'model_ar.npz'
np.savez(AR_FILE, W_ar=W_ar.astype(np.float32), b_ar=b_ar.astype(np.float32))

print(f"AutoRegressive model matrices saved to {AR_FILE}")
print("--- AR Model Cell Complete ---")


--- Training Model 4: AutoRegressive (AR) Model ---
AR Model Input Shape: (99999, 6)
AR Model Target Shape: (99999, 3)
Training AR(2) linear model...
AutoRegressive model matrices saved to model_ar.npz
--- AR Model Cell Complete ---


In [10]:
print("Calculating AR Model Mean Squared Error...")

# Get the AR model's predictions on the entire training dataset
y_pred_ar = ar_model.predict(X_ar_features)

# Calculate the MSE by comparing predictions to the true values
mse_ar = mean_squared_error(y_ar_target, y_pred_ar)

print(f"--- AR Model Training Results ---")
print(f"Mean Squared Error (MSE) on training data: {mse_ar:.10f}")

Calculating AR Model Mean Squared Error...
--- AR Model Training Results ---
Mean Squared Error (MSE) on training data: 0.0000065939


In [8]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.layers import Input, Dense, Conv1D, Lambda
from keras.models import Model

# ================================
# FILE CONFIG
# ================================
TCN_FILE = "model_tcn.h5"   # Your trained model
Q_FRAC = 16
SCALE = 1 << Q_FRAC

# ================================
# Q16.16 Quantization
# ================================
def q16(arr):
    q = np.round(arr * SCALE).astype(np.int64)
    q = np.clip(q, -2**31, 2**31 - 1).astype(np.int32)
    return q.astype(np.float32) / SCALE

# ================================
# REBUILD YOUR TINY-TCN ARCHITECTURE EXACTLY
# (Matches your training code)
# ================================
TCN_LOOKBACK = 4   # MUST MATCH training

def build_tcn():
    x_in = Input(shape=(TCN_LOOKBACK, 3))
    x = Conv1D(filters=8, kernel_size=2, padding='causal',
               activation='tanh', name="conv1")(x_in)
    x = Conv1D(filters=8, kernel_size=2, padding='causal',
               activation='tanh', name="conv2")(x)
    x = Lambda(lambda t: t[:, -1, :], name="get_last_timestep")(x)
    x_out = Dense(3, name="output_layer")(x)
    return Model(x_in, x_out, name="Tiny_TCN")

# ================================
# BUILD MODEL & LOAD WEIGHTS ONLY
# (No Lambda deserialization now)
# ================================
model = build_tcn()
model.load_weights(TCN_FILE)
print("✅ Successfully loaded weights from:", TCN_FILE)
model.summary()

# ================================
# GET LAYERS
# ================================
conv1 = model.get_layer("conv1")
conv2 = model.get_layer("conv2")
dense = model.get_layer("output_layer")

# ================================
# EXTRACT RAW KERAS WEIGHTS
# Keras Conv1D: (K, IN, OUT)
# Dense:        (IN, OUT)
# ================================
W1, B1 = conv1.get_weights()
W2, B2 = conv2.get_weights()
W3, B3 = dense.get_weights()

# ================================
# TRANSPOSE FOR VITIS HLS
# ================================
W_conv1 = np.transpose(W1, (2, 1, 0))   # (8, 3, 2)
W_conv2 = np.transpose(W2, (2, 1, 0))   # (8, 8, 2)
W_out   = np.transpose(W3, (1, 0))      # (3, 8)

# ================================
# QUANTIZATION
# ================================
W_conv1_q = q16(W_conv1)
B_conv1_q = q16(B1)

W_conv2_q = q16(W_conv2)
B_conv2_q = q16(B2)

W_out_q   = q16(W_out)
B_out_q   = q16(B3)

# ================================
# DIMENSIONS
# ================================
TCN_CH1, N_INPUT, TCN_K = W_conv1_q.shape
TCN_CH2, _, _ = W_conv2_q.shape
N_OUTPUT, _ = W_out_q.shape
TCN_WIN = TCN_LOOKBACK
N_STATE = TCN_CH2 * TCN_WIN

# ================================
# WRITE FINAL HLS HEADER
# ================================
with open("tcn_weights.h", "w") as f:
    f.write("#pragma once\n\n")
    f.write("#include <ap_fixed.h>\n")
    f.write("#include <ap_int.h>\n\n")
    f.write("typedef ap_fixed<32,16> data_t;\n\n")

    f.write(f"const int N_INPUT   = {N_INPUT};\n")
    f.write(f"const int N_OUTPUT  = {N_OUTPUT};\n")
    f.write(f"const int TCN_WIN   = {TCN_WIN};\n")
    f.write(f"const int TCN_CH1   = {TCN_CH1};\n")
    f.write(f"const int TCN_CH2   = {TCN_CH2};\n")
    f.write(f"const int TCN_K     = {TCN_K};\n")
    f.write(f"const int N_STATE  = {N_STATE};\n\n")

    # ----- CONV1 -----
    f.write("static const data_t W_conv1[TCN_CH1][N_INPUT][TCN_K] = {\n")
    for c in range(TCN_CH1):
        f.write("  {\n")
        for i in range(N_INPUT):
            f.write("    {")
            f.write(", ".join(f"{x:.8f}" for x in W_conv1_q[c,i]))
            f.write("}")
            if i != N_INPUT-1: f.write(",")
            f.write("\n")
        f.write("  }")
        if c != TCN_CH1-1: f.write(",")
        f.write("\n")
    f.write("};\n\n")

    f.write("static const data_t B_conv1[TCN_CH1] = { ")
    f.write(", ".join(f"{x:.8f}" for x in B_conv1_q))
    f.write(" };\n\n")

    # ----- CONV2 -----
    f.write("static const data_t W_conv2[TCN_CH2][TCN_CH1][TCN_K] = {\n")
    for c2 in range(TCN_CH2):
        f.write("  {\n")
        for i in range(TCN_CH1):
            f.write("    {")
            f.write(", ".join(f"{x:.8f}" for x in W_conv2_q[c2,i]))
            f.write("}")
            if i != TCN_CH1-1: f.write(",")
            f.write("\n")
        f.write("  }")
        if c2 != TCN_CH2-1: f.write(",")
        f.write("\n")
    f.write("};\n\n")

    f.write("static const data_t B_conv2[TCN_CH2] = { ")
    f.write(", ".join(f"{x:.8f}" for x in B_conv2_q))
    f.write(" };\n\n")

    # ----- OUTPUT -----
    f.write("static const data_t W_out[N_OUTPUT][TCN_CH2] = {\n")
    for o in range(N_OUTPUT):
        f.write("  {")
        f.write(", ".join(f"{x:.8f}" for x in W_out_q[o]))
        f.write("}")
        if o != N_OUTPUT-1: f.write(",")
        f.write("\n")
    f.write("};\n\n")

    f.write("static const data_t B_out[N_OUTPUT] = { ")
    f.write(", ".join(f"{x:.8f}" for x in B_out_q))
    f.write(" };\n")

print("\n✅✅ SUCCESS: tcn_weights.h generated safely for Vitis HLS ✅✅")


✅ Successfully loaded weights from: model_tcn.h5


Model: "Tiny_TCN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 4, 3)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1 (Conv1D)                       │ (None, 4, 8)                │              56 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2 (Conv1D)                       │ (None, 4, 8)                │             136 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ get_last_timestep (Lambda)           │ (None, 8)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ output_layer (Dense)                 │ (None, 3)                   │              27 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 219 (876.00 B)

 Trainable params: 219 (876.00 B)

 Non-trainable params: 0 (0.00 B)


✅✅ SUCCESS: tcn_weights.h generated safely for Vitis HLS ✅✅


In [9]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.layers import Input, Dense
from keras.models import Model

# ================================
# CONFIG
# ================================
MLP_FILE = "model_mlp.h5"   # your saved MLP model
Q_FRAC   = 16
SCALE    = 1 << Q_FRAC

# ================================
# Q16.16 quantization
# ================================
def q16(arr):
    q = np.round(arr * SCALE).astype(np.int64)
    q = np.clip(q, -2**31, 2**31 - 1).astype(np.int32)
    return q.astype(np.float32) / SCALE

# ================================
# Rebuild Tiny-MLP architecture
# (must exactly match training code)
# ================================
def build_mlp():
    x_in = Input(shape=(3,))
    x = Dense(4, activation='tanh', name="hidden_layer")(x_in)
    x_out = Dense(3, name="output_layer")(x)
    return Model(x_in, x_out, name="Tiny_MLP")

# Build and load weights only
model = build_mlp()
model.load_weights(MLP_FILE)
print("✅ Loaded MLP weights from:", MLP_FILE)
model.summary()

# ================================
# Extract layers & weights
# ================================
hidden = model.get_layer("hidden_layer")
out    = model.get_layer("output_layer")

W1, B1 = hidden.get_weights()   # W1: (3,4), B1: (4,)
W2, B2 = out.get_weights()      # W2: (4,3), B2: (3,)

# Transpose to HLS layout:
#  - HLS wants W1[HIDDEN][N_INPUT], W2[N_OUTPUT][HIDDEN]
W1_hls = W1.T   # (4,3)
W2_hls = W2.T   # (3,4)

# Quantise
W1_q = q16(W1_hls)
B1_q = q16(B1)
W2_q = q16(W2_hls)
B2_q = q16(B2)

# Dimensions
N_HIDDEN, N_INPUT = W1_q.shape
N_OUTPUT, _       = W2_q.shape

print("\nDims:")
print("  N_INPUT  =", N_INPUT)
print("  N_HIDDEN =", N_HIDDEN)
print("  N_OUTPUT =", N_OUTPUT)

# ================================
# Write mlp_weights.h
# ================================
with open("mlp_weights.h", "w") as f:
    f.write("#pragma once\n\n")
    f.write("#include <ap_fixed.h>\n")
    f.write("#include <ap_int.h>\n\n")
    f.write("typedef ap_fixed<32,16> data_t;\n\n")

    f.write(f"const int N_INPUT   = {N_INPUT};\n")
    f.write(f"const int N_HIDDEN  = {N_HIDDEN};\n")
    f.write(f"const int N_OUTPUT  = {N_OUTPUT};\n\n")

    # ---- hidden weights ----
    f.write("static const data_t W1[N_HIDDEN][N_INPUT] = {\n")
    for h in range(N_HIDDEN):
        f.write("  {")
        f.write(", ".join(f"{x:.8f}" for x in W1_q[h, :]))
        f.write("}")
        if h != N_HIDDEN - 1:
            f.write(",")
        f.write("\n")
    f.write("};\n\n")

    # ---- hidden biases ----
    f.write("static const data_t B1[N_HIDDEN] = { ")
    f.write(", ".join(f"{x:.8f}" for x in B1_q))
    f.write(" };\n\n")

    # ---- output weights ----
    f.write("static const data_t W2[N_OUTPUT][N_HIDDEN] = {\n")
    for o in range(N_OUTPUT):
        f.write("  {")
        f.write(", ".join(f"{x:.8f}" for x in W2_q[o, :]))
        f.write("}")
        if o != N_OUTPUT - 1:
            f.write(",")
        f.write("\n")
    f.write("};\n\n")

    # ---- output biases ----
    f.write("static const data_t B2[N_OUTPUT] = { ")
    f.write(", ".join(f"{x:.8f}" for x in B2_q))
    f.write(" };\n")

print("\n✅ SUCCESS: generated mlp_weights.h")


✅ Loaded MLP weights from: model_mlp.h5


Model: "Tiny_MLP"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 3)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ hidden_layer (Dense)                 │ (None, 4)                   │              16 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ output_layer (Dense)                 │ (None, 3)                   │              15 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 31 (124.00 B)

 Trainable params: 31 (124.00 B)

 Non-trainable params: 0 (0.00 B)


Dims:
  N_INPUT  = 3
  N_HIDDEN = 4
  N_OUTPUT = 3

✅ SUCCESS: generated mlp_weights.h


In [1]:
# This script FORCE-FIXES Unicode minus in-place

BAD = "\u2212"   # Unicode minus
GOOD = "-"       # ASCII minus

fname = "tcn_weights_q.vh"

with open(fname, "r", encoding="utf-8", errors="ignore") as f:
    data = f.read()

if BAD in data:
    print("Unicode minus found. Fixing...")
    data = data.replace(BAD, GOOD)
else:
    print("No Unicode minus found.")

# Write back as pure ASCII
with open(fname, "w", encoding="ascii", errors="ignore") as f:
    f.write(data)

print("Done. File rewritten with ASCII '-' only.")


No Unicode minus found.
Done. File rewritten with ASCII '-' only.


In [2]:
with open("tcn_weights_q.vh", "rb") as f:
    if b"\xe2\x88\x92" in f.read():
        print("❌ STILL BROKEN")
    else:
        print("✅ FIXED — ASCII ONLY")


✅ FIXED — ASCII ONLY


In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Conv1D, Lambda
from tensorflow.keras.models import Model

# =================================================
# CONFIG
# =================================================
TCN_FILE = "model_tcn.h5"
Q_FRAC = 16
SCALE = 1 << Q_FRAC
TCN_LOOKBACK = 4   # MUST MATCH TRAINING

# =================================================
# Q16.16 Quantization
# =================================================
def q16(x):
    q = np.round(x * SCALE).astype(np.int64)
    q = np.clip(q, -2**31, 2**31 - 1)
    return (q.astype(np.float32) / SCALE)

# =================================================
# Rebuild model EXACTLY (no Lambda serialization)
# =================================================
def build_tcn():
    x_in = Input(shape=(TCN_LOOKBACK, 3))
    x = Conv1D(8, 2, padding="causal", activation="tanh", name="conv1")(x_in)
    x = Conv1D(8, 2, padding="causal", activation="tanh", name="conv2")(x)
    x = Lambda(lambda t: t[:, -1, :], name="last")(x)
    y = Dense(3, name="out")(x)
    return Model(x_in, y)

model = build_tcn()
model.load_weights(TCN_FILE)
print("✅ Model loaded")

# =================================================
# Extract weights
# =================================================
W1, B1 = model.get_layer("conv1").get_weights()  # (K, IN, OUT)
W2, B2 = model.get_layer("conv2").get_weights()
W3, B3 = model.get_layer("out").get_weights()    # (IN, OUT)

# =================================================
# Transpose to HLS / PYNQ layout
# =================================================
W_conv1 = np.transpose(W1, (2, 1, 0))   # (8, 3, 2)
W_conv2 = np.transpose(W2, (2, 1, 0))   # (8, 8, 2)
W_out   = np.transpose(W3, (1, 0))      # (3, 8)

# =================================================
# Quantize
# =================================================
W_conv1_q = q16(W_conv1)
B_conv1_q = q16(B1)
W_conv2_q = q16(W_conv2)
B_conv2_q = q16(B2)
W_out_q   = q16(W_out)
B_out_q   = q16(B3)

# =================================================
# Dimensions
# =================================================
TCN_CH1, N_INPUT, TCN_K = W_conv1_q.shape
TCN_CH2 = W_conv2_q.shape[0]
N_OUTPUT = W_out_q.shape[0]
TCN_WIN = TCN_LOOKBACK

# =================================================
# WRITE PYTHON WEIGHTS (FOR PYNQ)
# =================================================
with open("tcn_weights.py", "w") as f:
    f.write("import numpy as np\n\n")
    f.write(f"N_INPUT={N_INPUT}\n")
    f.write(f"N_OUTPUT={N_OUTPUT}\n")
    f.write(f"TCN_WIN={TCN_WIN}\n")
    f.write(f"TCN_CH1={TCN_CH1}\n")
    f.write(f"TCN_CH2={TCN_CH2}\n")
    f.write(f"TCN_K={TCN_K}\n\n")

    f.write(f"W_conv1 = np.array({W_conv1_q.tolist()}, dtype=np.float32)\n")
    f.write(f"B_conv1 = np.array({B_conv1_q.tolist()}, dtype=np.float32)\n\n")
    f.write(f"W_conv2 = np.array({W_conv2_q.tolist()}, dtype=np.float32)\n")
    f.write(f"B_conv2 = np.array({B_conv2_q.tolist()}, dtype=np.float32)\n\n")
    f.write(f"W_out = np.array({W_out_q.tolist()}, dtype=np.float32)\n")
    f.write(f"B_out = np.array({B_out_q.tolist()}, dtype=np.float32)\n")

print("✅ Generated tcn_weights.py for PYNQ")



✅ Model loaded
✅ Generated tcn_weights.py for PYNQ


In [2]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.layers import Input, Dense
from keras.models import Model

# ================================
# CONFIG
# ================================
MLP_FILE = "model_mlp.h5"   # your trained model
Q_FRAC = 16
SCALE = 1 << Q_FRAC

# ================================
# Q16.16 Quantization
# ================================
def q16(x):
    q = np.round(x * SCALE)
    q = np.clip(q, -2**31, 2**31 - 1)
    return (q.astype(np.int32)).astype(np.float32) / SCALE

# ================================
# REBUILD MODEL (EXACT)
# ================================
def build_mlp():
    x_in = Input(shape=(3,))
    x = Dense(4, activation="tanh", name="hidden_layer")(x_in)
    x_out = Dense(3, name="output_layer")(x)
    return Model(x_in, x_out, name="Tiny_MLP")

model = build_mlp()
model.load_weights(MLP_FILE)

print("✅ Loaded MLP weights")
model.summary()

# ================================
# EXTRACT KERAS WEIGHTS
# Dense: (IN, OUT)
# ================================
W1k, B1k = model.get_layer("hidden_layer").get_weights()
W2k, B2k = model.get_layer("output_layer").get_weights()

# ================================
# TRANSPOSE FOR HLS / PYNQ
# ================================
W1 = W1k.T   # (4, 3)
W2 = W2k.T   # (3, 4)

# ================================
# QUANTIZE
# ================================
W1q = q16(W1)
B1q = q16(B1k)
W2q = q16(W2)
B2q = q16(B2k)

# ================================
# CONSTANTS
# ================================
N_INPUT  = 3
N_HIDDEN = 4
N_OUTPUT = 3

# ================================
# WRITE PYTHON WEIGHTS FILE
# ================================
with open("mlp_weights.py", "w") as f:
    f.write("import numpy as np\n\n")
    f.write(f"N_INPUT  = {N_INPUT}\n")
    f.write(f"N_HIDDEN = {N_HIDDEN}\n")
    f.write(f"N_OUTPUT = {N_OUTPUT}\n\n")

    f.write("W1 = np.array([\n")
    for h in range(N_HIDDEN):
        f.write("  [" + ", ".join(f"{x:.8f}" for x in W1q[h]) + "]")
        if h != N_HIDDEN - 1:
            f.write(",")
        f.write("\n")
    f.write("], dtype=np.float32)\n\n")

    f.write("B1 = np.array([\n  ")
    f.write(", ".join(f"{x:.8f}" for x in B1q))
    f.write("\n], dtype=np.float32)\n\n")

    f.write("W2 = np.array([\n")
    for o in range(N_OUTPUT):
        f.write("  [" + ", ".join(f"{x:.8f}" for x in W2q[o]) + "]")
        if o != N_OUTPUT - 1:
            f.write(",")
        f.write("\n")
    f.write("], dtype=np.float32)\n\n")

    f.write("B2 = np.array([\n  ")
    f.write(", ".join(f"{x:.8f}" for x in B2q))
    f.write("\n], dtype=np.float32)\n")

print("✅ mlp_weights.py generated successfully")


✅ Loaded MLP weights


Model: "Tiny_MLP"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 3)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ hidden_layer (Dense)                 │ (None, 4)                   │              16 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ output_layer (Dense)                 │ (None, 3)                   │              15 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 31 (124.00 B)

 Trainable params: 31 (124.00 B)

 Non-trainable params: 0 (0.00 B)

✅ mlp_weights.py generated successfully
